# Módulo 05 · Almacén vectorial

`VectorStore` guarda los embeddings en una matriz y resuelve cada búsqueda con un único
producto matriz-vector.

In [1]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR = ROOT / "data" / "novatech"

In [2]:
from fundamenta.chunking import chunk_documents, contextualize
from fundamenta.embeddings import SentenceTransformerEmbedder
from fundamenta.loaders import load_directory
from fundamenta.vector_store import VectorStore

chunks = chunk_documents(load_directory(DATA_DIR))
embedder = SentenceTransformerEmbedder()

store = VectorStore()
store.add(embedder.embed_documents([contextualize(c) for c in chunks]), chunks)
print(f"{len(store)} vectores de {store.vectors.shape[1]} dimensiones")

41 vectores de 384 dimensiones


## Búsqueda

In [3]:
def show(results):
    for i, score in results:
        doc = store.documents[i]
        label = doc.metadata.get("section") or doc.text[:60].replace(chr(10), " ")
        print(f"  {score:.3f}  {doc.source} | {label}")


show(store.search(embedder.embed_query("¿Cuánto me devuelven si voy en mi coche?"), k=3))

  0.846  politica-de-gastos-y-viajes.md | Política de gastos y viajes > Transporte
  0.825  beneficios-2026.pdf | 4. Bienestar NovaTech reembolsa hasta 30 € al mes de la cuot
  0.821  politica-de-gastos-y-viajes.md | Política de gastos y viajes > Comidas con clientes


## Filtros por metadatos

Los filtros permiten restringir la búsqueda, por ejemplo a un documento, a un
departamento o a los documentos que un usuario tiene permiso para consultar.

In [4]:
query = embedder.embed_query("¿Qué plazos hay que cumplir?")
print("Sin filtro:")
show(store.search(query, k=3))
print("Solo en el catálogo de productos:")
show(store.search(query, k=3, where={"source": "productos-y-precios.md"}))

Sin filtro:
  0.836  beneficios-2026.pdf | El presupuesto no utilizado no se acumula para el año siguie
  0.835  politica-de-gastos-y-viajes.md | Política de gastos y viajes > Principios generales
  0.830  manual-del-empleado.md | Manual del empleado de NovaTech > Permisos retribuidos
Solo en el catálogo de productos:
  0.819  productos-y-precios.md | Productos y precios de NovaTech > Política de cancelación
  0.808  productos-y-precios.md | Productos y precios de NovaTech > Soporte y SLA
  0.791  productos-y-precios.md | Productos y precios de NovaTech > Soporte y SLA


## Persistencia

El índice se guarda en disco para no recalcular los embeddings en cada arranque.

In [5]:
import tempfile

with tempfile.TemporaryDirectory() as directory:
    store.save(directory)
    loaded = VectorStore.load(directory)
    print("Archivos:", sorted(p.name for p in Path(directory).iterdir()))
    print("Vectores cargados:", loaded.vectors.shape)

Archivos: ['documents.json', 'vectors.npy']
Vectores cargados: (41, 384)


## Escalabilidad de la búsqueda exacta

Se mide el tiempo de búsqueda sobre colecciones sintéticas de distinto tamaño, con la
misma dimensión que el modelo E5 (384).

In [6]:
import time

import numpy as np

from fundamenta.loaders import Document

rng = np.random.default_rng(0)
for n in (1_000, 10_000, 100_000):
    synthetic = VectorStore()
    synthetic.add(rng.standard_normal((n, 384), dtype=np.float32), [Document("")] * n)
    q = rng.standard_normal(384, dtype=np.float32)
    start = time.perf_counter()
    for _ in range(20):
        synthetic.search(q, k=5)
    elapsed = (time.perf_counter() - start) / 20 * 1000
    print(f"{n:>8} vectores: {elapsed:7.2f} ms por búsqueda")

    1000 vectores:    0.09 ms por búsqueda


   10000 vectores:    0.64 ms por búsqueda


  100000 vectores:    4.57 ms por búsqueda


La búsqueda exacta resuelve cientos de miles de fragmentos en milisegundos. Las bases
de datos vectoriales con índices aproximados (HNSW, IVF) se justifican a partir de
varios millones de vectores o cuando se necesitan actualizaciones concurrentes,
replicación y control de acceso (módulo 11).